[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_TimesFM/VaR_CEE_TimesFM.ipynb)

# VaR_CEE_TimesFM

Generates zero-shot VaR and ES forecasts using Google TimesFM 2.5 foundation model.
Fits a Student-t distribution to the model's output quantiles via quantile matching
optimization and derives VaR/ES from the fitted parametric distribution.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

**R1 update:** context 512 (was 250), stride 1 (full daily run).

In [ ]:
!pip install -q timesfm

import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import t as t_dist

%matplotlib inline

MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"}
}
OOS_START = "2018-01-01"
VAR_ALPHAS = [0.01, 0.025, 0.05]
ES_ALPHA = 0.025
FM_CONTEXT = 512       # R1: was 250
FM_HORIZON = 1
STRIDE = 1             # R1: was 5
SEED = 42

PROC_DIR = Path("data/processed")
VAR_DIR = Path("data/var_forecasts")
VAR_DIR.mkdir(parents=True, exist_ok=True)

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Load Data from Parquet

In [ ]:
returns_dict = {}
for country, raw_id, series_id, stype in get_all_series():
    pq = PROC_DIR / f"{country}_returns.parquet"
    if not pq.exists():
        print(f"  WARNING: {pq} not found, skipping {series_id}")
        continue
    df = pd.read_parquet(pq)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    if series_id in df.columns:
        returns_dict[series_id] = df[series_id].dropna()
        print(f"  {series_id}: {len(returns_dict[series_id])} obs")
    else:
        print(f"  WARNING: {series_id} not in {pq.name}")
print(f"\nTotal: {len(returns_dict)} series")

## 2. Load TimesFM Model

In [ ]:
import timesfm

print("Loading TimesFM 2.5 model...")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)
forecast_config = timesfm.ForecastConfig(
    max_context=FM_CONTEXT,
    max_horizon=FM_HORIZON,
    per_core_batch_size=1,
    force_flip_invariance=True,
    infer_is_positive=False,
)
tfm.compile(forecast_config)
print(f"TimesFM 2.5 loaded (context={FM_CONTEXT})")

## 3. Run VaR/ES Forecasting

In [ ]:
def fit_t_to_quantiles(q_vals, q_levels):
    """Fit Student-t to quantile points via quantile matching."""
    def objective(params):
        loc, scale, df = params
        if scale <= 0 or df <= 2:
            return 1e10
        predicted = t_dist.ppf(q_levels, df=df, loc=loc, scale=scale)
        return np.sum((predicted - q_vals) ** 2)

    loc0 = np.median(q_vals)
    scale0 = max((q_vals[-1] - q_vals[0]) / 4, 1e-8)
    best = None
    best_val = np.inf
    for df0 in [3.0, 5.0, 10.0, 30.0]:
        res = minimize(objective, [loc0, scale0, df0],
                       method='Nelder-Mead', options={'maxiter': 2000})
        if res.fun < best_val:
            best_val = res.fun
            best = res.x
    return best


def run_timesfm(tfm, returns, oos_dates, series_id, save_path):
    """Run TimesFM VaR/ES forecasts with incremental saving."""
    results = []
    n_total = len(oos_dates)

    for i, date in enumerate(oos_dates):
        loc = returns.index.get_loc(date)
        if loc < FM_CONTEXT:
            continue
        context = returns.iloc[loc - FM_CONTEXT:loc].values.astype(np.float64)
        y_true = float(returns.iloc[loc])

        try:
            point_fc, quant_fc = tfm.forecast(horizon=FM_HORIZON, inputs=[context])
            qf = quant_fc[0, 0, :]
            row = {"date": str(date.date()), "y_true": y_true}

            if len(qf) >= 9:
                q_vals = qf[-9:] if len(qf) > 9 else qf
                q_levels = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

                params = fit_t_to_quantiles(q_vals, q_levels)
                loc_p, scale_p, df_p = params
                df_p = max(df_p, 2.01)
                scale_p = max(scale_p, 1e-10)

                for alpha in VAR_ALPHAS:
                    a = {0.01: "01", 0.025: "025", 0.05: "05"}[alpha]
                    row[f"var_{a}"] = float(t_dist.ppf(alpha, df=df_p,
                                                        loc=loc_p, scale=scale_p))

                var_es = t_dist.ppf(ES_ALPHA, df=df_p, loc=loc_p, scale=scale_p)
                t_val = (var_es - loc_p) / scale_p
                pdf_val = t_dist.pdf(t_val, df=df_p)
                row["es_025"] = float(loc_p + scale_p * (
                    -pdf_val / ES_ALPHA * (df_p + t_val**2) / (df_p - 1)
                ))
            else:
                pf = float(point_fc[0, 0])
                sigma = float(np.std(context[-60:]))
                for alpha in VAR_ALPHAS:
                    a = {0.01: "01", 0.025: "025", 0.05: "05"}[alpha]
                    row[f"var_{a}"] = pf + sigma * t_dist.ppf(alpha, df=5)
                row["es_025"] = pf + sigma * t_dist.ppf(ES_ALPHA, df=5) * 1.3

            row["fm_median"] = float(point_fc[0, 0])
            results.append(row)

        except Exception as e:
            if i < 5:
                print(f"    Error at {date}: {e}")
            continue

        if len(results) % 100 == 0:
            pd.DataFrame(results).to_csv(save_path, index=False, float_format="%.8f")

        if (i + 1) % 200 == 0 or i == n_total - 1:
            print(f"    [{i+1}/{n_total}] {series_id}")

    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False, float_format="%.8f")
    return df

In [ ]:
print("=" * 60)
print("TIMESFM 2.5 VaR/ES FORECASTING")
print(f"Context: {FM_CONTEXT} | Stride: {STRIDE}")
print("=" * 60)

all_results = {}
for country, raw_id, series_id, stype in get_all_series():
    if series_id not in returns_dict:
        continue
    returns = returns_dict[series_id]
    oos_mask = returns.index >= pd.Timestamp(OOS_START)
    oos_dates = returns.index[oos_mask][::STRIDE]

    out_path = VAR_DIR / f"TimesFM-2.5_{series_id}.csv"

    # Resume support
    if out_path.exists():
        existing = pd.read_csv(out_path)
        if len(existing) >= len(oos_dates):
            print(f"\n[{series_id}] Done ({len(existing)} rows), skipping.")
            all_results[series_id] = existing
            continue

    print(f"\n[{series_id}] {len(returns)} total obs, {len(oos_dates)} OOS dates")

    t0 = time.time()
    df_result = run_timesfm(tfm, returns, oos_dates, series_id, out_path)
    all_results[series_id] = df_result
    print(f"  Saved {out_path.name} ({len(df_result)} rows) in {time.time()-t0:.1f}s")

print("\n=== TimesFM 2.5 complete ===")

## 4. Visualisation

In [ ]:
for country, raw_id, series_id, stype in get_all_series():
    if stype != "index" or series_id not in all_results:
        continue
    df_plot = all_results[series_id].copy()
    if "date" in df_plot.columns:
        df_plot["date"] = pd.to_datetime(df_plot["date"])
    var_col = "var_01"
    if var_col not in df_plot.columns:
        continue
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_plot["date"], df_plot["y_true"], linewidth=0.4, color="black", label="Returns")
    ax.plot(df_plot["date"], df_plot[var_col], linewidth=0.6, color="coral", linestyle="--", label="TimesFM VaR 1%")
    breaches = df_plot["y_true"] < df_plot[var_col]
    if breaches.any():
        ax.scatter(df_plot.loc[breaches, "date"], df_plot.loc[breaches, "y_true"],
                   color="red", s=10, zorder=5, alpha=0.7, label="Breach")
    n_b = breaches.sum()
    ax.set_title(f"TimesFM 2.5 VaR 1% \u2014 {series_id} (breaches: {n_b}/{len(df_plot)} = {100*n_b/len(df_plot):.1f}%)")
    ax.set_ylabel("Log returns")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
    ax.axhline(y=0, color="gray", linewidth=0.3)
    plt.tight_layout()
    plt.show()

## 5. Download Results

In [ ]:
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(VAR_DIR.glob("TimesFM-2.5_*.csv")):
        zf.write(f, f.name)
zip_buffer.seek(0)
with open("TimesFM_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())

# On Colab: uncomment to download
# from google.colab import files
# files.download("TimesFM_results.zip")

print(f"Saved TimesFM_results.zip ({len(list(VAR_DIR.glob('TimesFM-2.5_*.csv')))} files)")